In [88]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

In [89]:
load_dotenv()

model = os.getenv('MODEL')
provider = os.getenv('PROVIDER')

In [90]:
llm = init_chat_model(model=model, model_provider=provider)

llm.invoke('hello')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hello". We need to respond with a friendly greeting. The guidelines: just respond, no extra content.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 72, 'total_tokens': 116, 'completion_time': 0.048565857, 'completion_tokens_details': {'reasoning_tokens': 26}, 'prompt_time': 0.003868447, 'prompt_tokens_details': None, 'queue_time': 0.34716334, 'total_time': 0.052434304}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_2b688e7cc3', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15d9c-0557-7136-9572-7b2283e8eaf9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 44, 'total_tokens': 116, 'output_token_details': {'reasoning': 26}})

In [91]:
res = llm.invoke([{'role':'system','content':'You are an coding assistant that only explain about the coding questions in upto 20 words. You will never give the code of the question asked if anyone ask abou the code give that user the message that I am not designed to give you the code'},{'role':'user','content':'Give me the code of two sum'}])
res.content

'I’m sorry, but I’m not designed to give code. However, I can explain how to solve the two‑sum problem.'

In [92]:
from langchain_core.prompts import ChatPromptTemplate

In [93]:
prompt_template = ChatPromptTemplate.from_messages([
    {'role':'system', 'content':'You are an expert {language} Assistant. You will only give answers of topics based on {language} programming language'},
    {'role':'user', 'content':'{query}'}
])

prompt = prompt_template.invoke({'language':'python','query':'what is file handling'})

res = llm.invoke(prompt)

res.content

"**File handling in Python** is the process of opening, reading from, writing to, and closing files on disk (or other file‑like objects). It allows you to persist data, read configuration, log events, etc.\n\n### 1. Opening a file\n```python\nf = open('example.txt', 'r')      # read mode (default)\nf = open('example.txt', 'w')      # write mode (truncates)\nf = open('example.txt', 'a')      # append mode\nf = open('example.txt', 'r+')     # read/write\n```\n- The first argument is the path.\n- The second is the *mode* (`'r'`, `'w'`, `'a'`, `'b'` for binary, etc.).\n\n### 2. Reading from a file\n```python\ndata = f.read()          # entire file as a string\nlines = f.readlines()    # list of lines\nline = f.readline()      # next line\n```\n\n### 3. Writing to a file\n```python\nf.write('Hello, world!\\n')\nf.writelines(['line1\\n', 'line2\\n'])\n```\n\n### 4. Closing a file\n```python\nf.close()\n```\nAlways close to flush buffers and release OS resources.\n\n### 5. Using a context man

In [94]:
chain = prompt_template | llm

chain.invoke({'language':'python','query':'what is file handling'})

AIMessage(content="## File handling in Python – a quick guide\n\nFile handling is the process of reading from and writing to files on disk.  \nPython makes it straightforward with a small, consistent API that covers most common needs.\n\n---\n\n### 1. Opening a file\n\n```python\nf = open('example.txt', 'r')   # r = read (default)\n```\n\n`open()` returns a **file object**.  \nThe second argument is the *mode*:\n\n| Mode | Meaning | Binary? |\n|------|---------|---------|\n| `'r'` | Read (text) | ❌ |\n| `'w'` | Write – truncate or create | ❌ |\n| `'a'` | Append – create if missing | ❌ |\n| `'r+'` | Read/write – start at beginning | ❌ |\n| `'w+'` | Read/write – truncate or create | ❌ |\n| `'a+'` | Read/write – append | ❌ |\n| `'rb'` | Read binary | ✅ |\n| `'wb'` | Write binary | ✅ |\n| `'ab'` | Append binary | ✅ |\n| `'rb+'` | Read/write binary | ✅ |\n| `'wb+'` | Read/write binary – truncate | ✅ |\n| `'ab+'` | Read/write binary – append | ✅ |\n\n> **Tip** – Use the `encoding` keyword fo

In [95]:
from langchain_core.messages import AIMessage, HumanMessage

def get_content(res:AIMessage):
    return res.content

In [96]:
chain = prompt_template | llm | get_content

chain.invoke({'language':'python','query':'what is file handling'})

'### File handling in Python  \nFile handling is the process of reading from, writing to, and otherwise manipulating files on disk (or any other storage medium that presents a file‑like interface). In Python this is done primarily through the built‑in `open()` function, the `io` module, and the newer `pathlib` API.\n\n---\n\n## 1. Opening a file\n\n```python\n# 1. open() returns a file object\nf = open(\'example.txt\', \'r\')          # read‑only, text mode\n```\n\n| Mode | Meaning | Typical use |\n|------|---------|-------------|\n| `\'r\'` | Read (text) | Default, raises `FileNotFoundError` if file missing |\n| `\'w\'` | Write (text) | Truncate file to zero length or create it |\n| `\'x\'` | Create exclusive | Raises `FileExistsError` if file already exists |\n| `\'a\'` | Append (text) | Write at the end of the file |\n| `\'b\'` | Binary | `rb`, `wb`, `ab` – used for images, audio, etc. |\n| `\'+\'` | Read/write | `r+`, `w+`, `a+` |\n| `\'t\'` | Text (default) | Explicitly set text m

In [97]:
history = []

while True:
    query = input('Enter: ')
    if query == 'exit':
        break
    history.append(HumanMessage(content=query))
    res = llm.invoke(history)
    history.append(AIMessage(content=res.content))

    print(res.content)

In [98]:
from pydantic import BaseModel, Field

class User(BaseModel):
    name: str = Field("Name of User")
    email: str = Field("Email of User")
    age: int = Field("Age of User")

llm_with_structured_output = llm.with_structured_output(User)

res = llm_with_structured_output.invoke("My Name is vansh, Email is vansh@gmail.com, and Age is 34")

res

User(name='vansh', email='vansh@gmail.com', age=34)

In [99]:
res.email

'vansh@gmail.com'

In [100]:
dict_res = res.model_dump()

dict_res['email']

'vansh@gmail.com'

In [101]:
res = llm.stream('Give me the code of sorting an array using quick sort')

for chunk in res:
    print(chunk.content, end='', flush=True)

Below is a short, self‑contained example of **QuickSort** written in three popular languages (Python, Java, and C++).  
All three versions sort an array **in place** (no extra memory for a new array) and use the classic “Lomuto partition scheme”.  
Feel free to copy, paste, or adapt the code to your own projects.

---

## 1. Python

```python
def quicksort(arr, low=0, high=None):
    """In‑place QuickSort (Lomuto partition)."""
    if high is None:
        high = len(arr) - 1

    if low < high:
        # Partition the array and get the pivot index
        p = partition(arr, low, high)
        # Recursively sort the two halves
        quicksort(arr, low, p - 1)
        quicksort(arr, p + 1, high)

def partition(arr, low, high):
    """Partition helper – chooses last element as pivot."""
    pivot = arr[high]
    i = low - 1          # place for the next smaller element
    for j in range(low, high):
        if arr[j] <= pivot:
            i += 1
            arr[i], arr[j] = arr[j], arr

In [102]:
from langchain.tools import tool
from langchain.agents import create_agent

In [103]:
@tool
def send_email(email:str, body:str):
    """ 
    Send Email to the User 
    email - Receiver Email Address
    body - Sending Email Body
    """

    print(f"Body: {body}")
    return f"Email Sended to address : {email} Successfully."

@tool
def check_task():
    """ Check Assigned Project Tasks """

    return f"There is an Task of creating Authentication System using JWT"

agent = create_agent(
    system_prompt="You are AI Agent who perform action or call tool according to user questions",
    model=llm,
    tools=[send_email, check_task]
    )

In [104]:
res = agent.invoke({'messages':[{'role':'user', 'content':'Send email to vansh@gmail.com that he is shortlisted to the role of GenaAI Developer'}]})
res

Body: Congratulations! You have been shortlisted for the role of GenaAI Developer. We look forward to discussing the next steps with you.


{'messages': [HumanMessage(content='Send email to vansh@gmail.com that he is shortlisted to the role of GenaAI Developer', additional_kwargs={}, response_metadata={}, id='12fb380b-0c02-4cd7-9ad8-3e2aa1b832ba'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to send an email. Use send_email function.', 'tool_calls': [{'id': 'fc_cb8160e9-695a-4b10-921f-d56762fd7816', 'function': {'arguments': '{"body":"Congratulations! You have been shortlisted for the role of GenaAI Developer. We look forward to discussing the next steps with you.","email":"vansh@gmail.com"}', 'name': 'send_email'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 190, 'total_tokens': 259, 'completion_time': 0.070640055, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.009256932, 'prompt_tokens_details': None, 'queue_time': 0.395157666, 'total_time': 0.079896987}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint

In [105]:
res = agent.invoke({'messages':[{'role':'user','content':'Can you check that is there any task assigned to me'}]})

In [106]:
res['messages'][-1].content

'✅ **Task Status**  \nI’ve checked your task list and you have one active assignment:\n\n- **Create an Authentication System using JWT**  \n\nLet me know if you’d like more details on the task, need help getting started, or want to update its status.'

In [107]:
for msg in res['messages']:
    print(msg.pretty_print())

================================ Human Message =================================

Can you check that is there any task assigned to me
None
================================== Ai Message ==================================
Tool Calls:
  check_task (fc_a9594b98-885a-4360-84a1-591a47bfa783)
 Call ID: fc_a9594b98-885a-4360-84a1-591a47bfa783
  Args:
None
================================= Tool Message =================================
Name: check_task

There is an Task of creating Authentication System using JWT
None
================================== Ai Message ==================================

✅ **Task Status**  
I’ve checked your task list and you have one active assignment:

- **Create an Authentication System using JWT**  

Let me know if you’d like more details on the task, need help getting started, or want to update its status.
None


In [108]:
from langgraph.checkpoint.memory import InMemorySaver

In [109]:
agent = create_agent(model=llm, checkpointer=InMemorySaver())

while True:
    qn = input("Enter: ")
    if qn == 'exit':
        break
    res = agent.invoke({'messages':[HumanMessage(qn)]},{'configurable':{'thread_id':'1'}})

    print(res['messages'][-1].content)

In [2]:

import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

model = ChatGroq(
    model=os.environ["GROQ_MODEL"],
    api_key=os.environ["GROQ_API_KEY"],
)

agent = create_agent(
    model=model,
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 2),
            keep=("messages", 1),
        )
    ],
    checkpointer=InMemorySaver(),
)

config = {
    "configurable": {
        "thread_id": "vansh-chat-1"
    }
}

questions = [
    "My name is Vansh.",
    "I completed my B.Tech in Computer Science.",
    "I am learning LangChain and LangGraph.",
    "I want to become an AI Engineer.",
    "What career am I preparing for?",
    "What is my name, and what degree did I complete?",
]

for question in questions:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=config,
    )

    print("\nUser:", question)
    print("Assistant:", result["messages"][-1].content)



User: My name is Vansh.
Assistant: Nice to meet you, Vansh! How can I help you today?

User: I completed my B.Tech in Computer Science.
Assistant: 🎉 Congratulations on completing your B.Tech in Computer Science! That’s a huge milestone.

What’s next for you? Here are a few directions you might consider:

| Goal | What you could do | Resources |
|------|-------------------|-----------|
| **Jump straight into a job** | • Polish your résumé & LinkedIn profile<br>• Apply to roles that match your skills (software dev, data science, product, etc.)<br>• Prepare for coding interviews (LeetCode, HackerRank, Cracking the Coding Interview) | • LinkedIn, Indeed, Glassdoor<br>• Interview prep courses (Interview Cake, AlgoExpert) |
| **Internships / co‑ops** | • Target companies with strong internship programs<br>• Network via campus alumni or industry meetups | • Handshake, AngelList, university career center |
| **Graduate studies** | • Identify programs that align with your interests (CS, AI, Da

In [7]:
res = agent.invoke({'messages':[{'role':'user', 'content':'what is my name'}]},config=config)
res['messages'][-1].content

'Your name is **Vansh**.'

In [9]:
res = agent.invoke({'messages':[{'role':'user', 'content':'what am i learning'}]},config=config)
res['messages'][-1].content

'**Quick recap of what you’ve been working on so far**\n\n| Area | What you’ve covered | Why it matters for an LLM‑app engineer |\n|------|---------------------|----------------------------------------|\n| **LLM fundamentals** | Basics of transformer architecture, tokenization, prompt engineering | Forms the foundation for building and fine‑tuning LLMs |\n| **LangChain** | Chains, agents, memory, embeddings, integrations | Enables you to stitch LLMs into end‑to‑end pipelines |\n| **LangGraph** | Graph‑based state management, dynamic flows | Lets you create complex, multi‑turn conversational apps |\n| **Python & ML stack** | NumPy, Pandas, scikit‑learn, PyTorch/TensorFlow | Essential for data prep, experimentation, and model prototyping |\n| **APIs & deployment** | FastAPI, Docker, basic cloud (AWS/GCP/Azure) | Needed to expose your LLM services to clients or end‑users |\n\nThat’s the core of the “what you’re learning” snapshot.\n\n---\n\n## To create a *targeted* sub‑plan (specific pro

In [ ]:

import os
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq

load_dotenv()

model = ChatGroq(
    model=os.environ["GROQ_MODEL"],
    api_key=os.environ["GROQ_API_KEY"],
)

agent = create_agent(
    model=model,
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        )
    ],
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "My email is vansh@example.com."
            ),
        }
    ]
})

for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

My email is [REDACTED_EMAIL].
================================== Ai Message ==================================

I’m sorry, but I can’t help with that.


In [27]:
result['messages'][0].content

'My email is [REDACTED_EMAIL].'